In [3]:
# 1-dataset model (HTCas9)

In [4]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import numpy as np

from torch.utils.data import Dataset, DataLoader, Subset
from sklearn.model_selection import train_test_split
from scipy.stats import spearmanr
import optuna

import copy

# PyG
from torch_geometric.data import Data, Batch
from torch_geometric.nn import GCNConv, global_mean_pool

# NUPACK
from nupack import Model, pairs

np.set_printoptions(threshold=np.inf)
print("Using PyTorch and PyG.")

########################################
# 1. Data Loading
########################################

def load_branch1_data_AsCas12aRVR():
    """
    Loads (60,4) data from "Feature_CNN1_reduced.txt".
    """
    data_branch1 = []
    current_array = []
    with open("Feature_CNN1_reduced.txt", 'r') as f:
        for line in f:
            line = line.strip()
            if line == "":
                if current_array:
                    data_branch1.append(current_array)
                    current_array = []
            else:
                line = line.replace('[','').replace(']', '')
                row = [float(x.strip().replace(',', '')) for x in line.split()]
                current_array.append(row)
    if current_array:
        data_branch1.append(current_array)
    X_branch1 = np.array(data_branch1)
    X_branch1 = X_branch1.reshape(len(X_branch1), 60, 4)
    return X_branch1

def load_reaction_rates_AsCas12aRVR():
    with open('AsCas12aRVR_indel_frequency_TTTV_filtered.txt', 'r') as f:
        rates = [float(line.strip()) for line in f.readlines()]
    return np.array(rates)


# NUPACK models
from nupack import Model, pairs
my_model_RNA = Model(material='rna', ensemble='nostacking', celsius=37, sodium=0.1, magnesium=0.01)
my_model_DNA = Model(material='dna', ensemble='nostacking', celsius=37, sodium=0.1, magnesium=0.01)

def DNA_reverse_complement(DNA):
    complement = {'A': 'T', 'C': 'G', 'G': 'C', 'T': 'A'}
    return ''.join(complement.get(base, base) for base in reversed(DNA))


########################################
# 3. CNN1 Branches (One hidden MLP)
########################################

class CNNBranch1(nn.Module):
    """
    CNN branch for (60,4).
    """
    def __init__(self, filters, kernel_size, dense_units):
        super().__init__()
        self.conv = nn.Conv1d(4, filters, kernel_size, padding=(kernel_size-1)//2)
        self.pool = nn.MaxPool1d(2)
        self.fc = nn.Linear(filters*30, dense_units)
    def forward(self, x):
        x = x.permute(0,2,1)  # (batch,4,30)
        x = F.relu(self.conv(x))
        x = self.pool(x)      # (batch,filters,30)
        x = x.flatten(start_dim=1)
        x = F.relu(self.fc(x))
        return x


########################################
# 4. Final Fusion Model
########################################

class CNN1OnlyModel(nn.Module):
    def __init__(self, filters1, kernel_size1, dense_units1, final_fc_dim, dropout_rate=0.0):
        super().__init__()
        self.cnn_branch1 = CNNBranch1(filters1, kernel_size1, dense_units1)
        self.dropout_rate = dropout_rate
        self.hidden = nn.Linear(dense_units1, final_fc_dim)
        self.out = nn.Linear(final_fc_dim, 1)

    def forward(self, x1):
        feat1 = self.cnn_branch1(x1)
        merged = F.dropout(feat1, p=self.dropout_rate, training=self.training)
        x = F.relu(self.hidden(merged))
        return self.out(x).view(-1)
        
########################################
# 5. Hybrid Dataset
########################################

class HybridDataset(Dataset):
    def __init__(self, X1, reaction_rates):
        super().__init__()
        self.X1 = X1
        self.reaction_rates = reaction_rates
        self.num_samples = len(reaction_rates)

    def __len__(self):
        return self.num_samples

    def __getitem__(self, idx):
        x1 = torch.tensor(self.X1[idx], dtype=torch.float)
        y_val = torch.tensor([self.reaction_rates[idx]], dtype=torch.float)
        return x1, y_val


def hybrid_collate(batch):
    x1_list, y_list = zip(*batch)
    x1 = torch.stack(x1_list, dim=0)
    y = torch.tensor(y_list, dtype=torch.float).view(-1)
    return x1, y


########################################
# 7. Main Pipeline
########################################

def main_pipeline():
    
    # List all model files you want to evaluate
    model_paths = [
        "../../../CNN1_only_models/trial_0_best_1_dataset_trained_CNN1.pt",
        "../../../CNN1_only_models/trial_1_best_1_dataset_trained_CNN1.pt",
        "../../../CNN1_only_models/trial_2_best_1_dataset_trained_CNN1.pt",
        "../../../CNN1_only_models/trial_3_best_1_dataset_trained_CNN1.pt",
        "../../../CNN1_only_models/trial_4_best_1_dataset_trained_CNN1.pt",
        "../../../CNN1_only_models/trial_5_best_1_dataset_trained_CNN1.pt",
        "../../../CNN1_only_models/trial_6_best_1_dataset_trained_CNN1.pt",
        "../../../CNN1_only_models/trial_7_best_1_dataset_trained_CNN1.pt",
        "../../../CNN1_only_models/trial_8_best_1_dataset_trained_CNN1.pt",
        "../../../CNN1_only_models/trial_9_best_1_dataset_trained_CNN1.pt"
    ]
    
    # final hold-out evaluation
    # load data

    X1_AsCas12aRVR = load_branch1_data_AsCas12aRVR()
    X1_AsCas12aRVR   = np.asarray(X1_AsCas12aRVR)
    X1 = np.concatenate([X1_AsCas12aRVR], axis=0) 

    rates_AsCas12aRVR = load_reaction_rates_AsCas12aRVR()
    rates_AsCas12aRVR   = np.asarray(rates_AsCas12aRVR)
    rates = np.concatenate([rates_AsCas12aRVR], axis=0) 

    hybrid_dataset = HybridDataset(X1, rates)

    # Get unseen AsCas12aRVR dataset
    np.random.seed(42)
    full_indices_AsCas12aRVR = np.arange(len(rates_AsCas12aRVR))
    selected_indices_AsCas12aRVR = np.random.choice(len(full_indices_AsCas12aRVR), size=len(full_indices_AsCas12aRVR), replace=False)
    unseen_indices_AsCas12aRVR = np.setdiff1d(full_indices_AsCas12aRVR, selected_indices_AsCas12aRVR)

    from torch.utils.data import ConcatDataset, DataLoader

    # Evaluate on 10 subsamples from unseen
    unseen_set_AsCas12aRVR = Subset(hybrid_dataset, unseen_indices_AsCas12aRVR)
    
    from torch.utils.data import ConcatDataset, DataLoader

    selected_set_AsCas12aRVR = Subset(hybrid_dataset, selected_indices_AsCas12aRVR)
    trial_loader = DataLoader(selected_set_AsCas12aRVR, batch_size=1, shuffle=False, collate_fn=hybrid_collate)
    
    spearman_results = []

    for model_path in model_paths:
        model = torch.load(model_path, weights_only=False)
        model.eval()

        preds_trial = []
        labels_trial = []

        with torch.no_grad():
            for xx1_b, yy_b in trial_loader:
                p = model(xx1_b)
                preds_trial.append(p.item())
                labels_trial.append(yy_b.item())

        sp_corr, _ = spearmanr(labels_trial, preds_trial)

        print(f"{model_path}: Spearman = {sp_corr}")
        spearman_results.append((model_path, sp_corr))

    with open("Spearman_all_models_1_dataset_CNN1.txt", "w") as f:
        for _, sp_corr in spearman_results:
            f.write(f"{sp_corr}\n")
    

if __name__ == "__main__":
    main_pipeline() 


Using PyTorch and PyG.
../../../CNN1_only_models/trial_0_best_1_dataset_trained_CNN1.pt: Spearman = 0.3527915702637233
../../../CNN1_only_models/trial_1_best_1_dataset_trained_CNN1.pt: Spearman = 0.38133233847743414
../../../CNN1_only_models/trial_2_best_1_dataset_trained_CNN1.pt: Spearman = 0.3658918909799705
../../../CNN1_only_models/trial_3_best_1_dataset_trained_CNN1.pt: Spearman = 0.35708097650986076
../../../CNN1_only_models/trial_4_best_1_dataset_trained_CNN1.pt: Spearman = 0.35790807096191224
../../../CNN1_only_models/trial_5_best_1_dataset_trained_CNN1.pt: Spearman = 0.3541954397500303
../../../CNN1_only_models/trial_6_best_1_dataset_trained_CNN1.pt: Spearman = 0.33751285870722186
../../../CNN1_only_models/trial_7_best_1_dataset_trained_CNN1.pt: Spearman = 0.3644509280048983
../../../CNN1_only_models/trial_8_best_1_dataset_trained_CNN1.pt: Spearman = 0.3430084933073402
../../../CNN1_only_models/trial_9_best_1_dataset_trained_CNN1.pt: Spearman = 0.34378469769779757


In [5]:
# 2-dataset model (HTCas9+HT11)

In [6]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import numpy as np

from torch.utils.data import Dataset, DataLoader, Subset
from sklearn.model_selection import train_test_split
from scipy.stats import spearmanr
import optuna

import copy

# PyG
from torch_geometric.data import Data, Batch
from torch_geometric.nn import GCNConv, global_mean_pool

# NUPACK
from nupack import Model, pairs

np.set_printoptions(threshold=np.inf)
print("Using PyTorch and PyG.")

########################################
# 1. Data Loading
########################################

def load_branch1_data_AsCas12aRVR():
    """
    Loads (60,4) data from "Feature_CNN1_reduced.txt".
    """
    data_branch1 = []
    current_array = []
    with open("Feature_CNN1_reduced.txt", 'r') as f:
        for line in f:
            line = line.strip()
            if line == "":
                if current_array:
                    data_branch1.append(current_array)
                    current_array = []
            else:
                line = line.replace('[','').replace(']', '')
                row = [float(x.strip().replace(',', '')) for x in line.split()]
                current_array.append(row)
    if current_array:
        data_branch1.append(current_array)
    X_branch1 = np.array(data_branch1)
    X_branch1 = X_branch1.reshape(len(X_branch1), 60, 4)
    return X_branch1

def load_reaction_rates_AsCas12aRVR():
    with open('AsCas12aRVR_indel_frequency_TTTV_filtered.txt', 'r') as f:
        rates = [float(line.strip()) for line in f.readlines()]
    return np.array(rates)


# NUPACK models
from nupack import Model, pairs
my_model_RNA = Model(material='rna', ensemble='nostacking', celsius=37, sodium=0.1, magnesium=0.01)
my_model_DNA = Model(material='dna', ensemble='nostacking', celsius=37, sodium=0.1, magnesium=0.01)

def DNA_reverse_complement(DNA):
    complement = {'A': 'T', 'C': 'G', 'G': 'C', 'T': 'A'}
    return ''.join(complement.get(base, base) for base in reversed(DNA))


########################################
# 3. CNN1 Branches (One hidden MLP)
########################################

class CNNBranch1(nn.Module):
    """
    CNN branch for (60,4).
    """
    def __init__(self, filters, kernel_size, dense_units):
        super().__init__()
        self.conv = nn.Conv1d(4, filters, kernel_size, padding=(kernel_size-1)//2)
        self.pool = nn.MaxPool1d(2)
        self.fc = nn.Linear(filters*30, dense_units)
    def forward(self, x):
        x = x.permute(0,2,1)  # (batch,4,30)
        x = F.relu(self.conv(x))
        x = self.pool(x)      # (batch,filters,30)
        x = x.flatten(start_dim=1)
        x = F.relu(self.fc(x))
        return x


########################################
# 4. Final Fusion Model
########################################

class CNN1OnlyModel(nn.Module):
    def __init__(self, filters1, kernel_size1, dense_units1, final_fc_dim, dropout_rate=0.0):
        super().__init__()
        self.cnn_branch1 = CNNBranch1(filters1, kernel_size1, dense_units1)
        self.dropout_rate = dropout_rate
        self.hidden = nn.Linear(dense_units1, final_fc_dim)
        self.out = nn.Linear(final_fc_dim, 1)

    def forward(self, x1):
        feat1 = self.cnn_branch1(x1)
        merged = F.dropout(feat1, p=self.dropout_rate, training=self.training)
        x = F.relu(self.hidden(merged))
        return self.out(x).view(-1)
        
########################################
# 5. Hybrid Dataset
########################################

class HybridDataset(Dataset):
    def __init__(self, X1, reaction_rates):
        super().__init__()
        self.X1 = X1
        self.reaction_rates = reaction_rates
        self.num_samples = len(reaction_rates)

    def __len__(self):
        return self.num_samples

    def __getitem__(self, idx):
        x1 = torch.tensor(self.X1[idx], dtype=torch.float)
        y_val = torch.tensor([self.reaction_rates[idx]], dtype=torch.float)
        return x1, y_val


def hybrid_collate(batch):
    x1_list, y_list = zip(*batch)
    x1 = torch.stack(x1_list, dim=0)
    y = torch.tensor(y_list, dtype=torch.float).view(-1)
    return x1, y


########################################
# 7. Main Pipeline
########################################

def main_pipeline():
    
    # List all model files you want to evaluate
    model_paths = [
        "../../../CNN1_only_models/trial_0_best_2_dataset_trained_CNN1.pt",
        "../../../CNN1_only_models/trial_1_best_2_dataset_trained_CNN1.pt",
        "../../../CNN1_only_models/trial_2_best_2_dataset_trained_CNN1.pt",
        "../../../CNN1_only_models/trial_3_best_2_dataset_trained_CNN1.pt",
        "../../../CNN1_only_models/trial_4_best_2_dataset_trained_CNN1.pt",
        "../../../CNN1_only_models/trial_5_best_2_dataset_trained_CNN1.pt",
        "../../../CNN1_only_models/trial_6_best_2_dataset_trained_CNN1.pt",
        "../../../CNN1_only_models/trial_7_best_2_dataset_trained_CNN1.pt",
        "../../../CNN1_only_models/trial_8_best_2_dataset_trained_CNN1.pt",
        "../../../CNN1_only_models/trial_9_best_2_dataset_trained_CNN1.pt"
    ]
    
    # final hold-out evaluation
    # load data

    X1_AsCas12aRVR = load_branch1_data_AsCas12aRVR()
    X1_AsCas12aRVR   = np.asarray(X1_AsCas12aRVR)
    X1 = np.concatenate([X1_AsCas12aRVR], axis=0) 

    rates_AsCas12aRVR = load_reaction_rates_AsCas12aRVR()
    rates_AsCas12aRVR   = np.asarray(rates_AsCas12aRVR)
    rates = np.concatenate([rates_AsCas12aRVR], axis=0) 

    hybrid_dataset = HybridDataset(X1, rates)

    # Get unseen AsCas12aRVR dataset
    np.random.seed(42)
    full_indices_AsCas12aRVR = np.arange(len(rates_AsCas12aRVR))
    selected_indices_AsCas12aRVR = np.random.choice(len(full_indices_AsCas12aRVR), size=len(full_indices_AsCas12aRVR), replace=False)
    unseen_indices_AsCas12aRVR = np.setdiff1d(full_indices_AsCas12aRVR, selected_indices_AsCas12aRVR)

    from torch.utils.data import ConcatDataset, DataLoader

    # Evaluate on 10 subsamples from unseen
    unseen_set_AsCas12aRVR = Subset(hybrid_dataset, unseen_indices_AsCas12aRVR)
    
    from torch.utils.data import ConcatDataset, DataLoader

    selected_set_AsCas12aRVR = Subset(hybrid_dataset, selected_indices_AsCas12aRVR)
    trial_loader = DataLoader(selected_set_AsCas12aRVR, batch_size=1, shuffle=False, collate_fn=hybrid_collate)
    
    spearman_results = []

    for model_path in model_paths:
        model = torch.load(model_path, weights_only=False)
        model.eval()

        preds_trial = []
        labels_trial = []

        with torch.no_grad():
            for xx1_b, yy_b in trial_loader:
                p = model(xx1_b)
                preds_trial.append(p.item())
                labels_trial.append(yy_b.item())

        sp_corr, _ = spearmanr(labels_trial, preds_trial)

        print(f"{model_path}: Spearman = {sp_corr}")
        spearman_results.append((model_path, sp_corr))

    with open("Spearman_all_models_2_dataset_CNN1.txt", "w") as f:
        for _, sp_corr in spearman_results:
            f.write(f"{sp_corr}\n")
    

if __name__ == "__main__":
    main_pipeline() 


Using PyTorch and PyG.
../../../CNN1_only_models/trial_0_best_2_dataset_trained_CNN1.pt: Spearman = 0.4822031679998469
../../../CNN1_only_models/trial_1_best_2_dataset_trained_CNN1.pt: Spearman = 0.5162603516150719
../../../CNN1_only_models/trial_2_best_2_dataset_trained_CNN1.pt: Spearman = 0.48681308336501455
../../../CNN1_only_models/trial_3_best_2_dataset_trained_CNN1.pt: Spearman = 0.4874637088232509
../../../CNN1_only_models/trial_4_best_2_dataset_trained_CNN1.pt: Spearman = 0.49669504636563483
../../../CNN1_only_models/trial_5_best_2_dataset_trained_CNN1.pt: Spearman = 0.49587393356731213
../../../CNN1_only_models/trial_6_best_2_dataset_trained_CNN1.pt: Spearman = 0.5046364813276761
../../../CNN1_only_models/trial_7_best_2_dataset_trained_CNN1.pt: Spearman = 0.5063565622082733
../../../CNN1_only_models/trial_8_best_2_dataset_trained_CNN1.pt: Spearman = 0.46233271046466107
../../../CNN1_only_models/trial_9_best_2_dataset_trained_CNN1.pt: Spearman = 0.48528591614520566


In [7]:
# 4-dataset model (HTCas9+HT11+RfxCas13d(mmc4+mmc5))

In [8]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import numpy as np

from torch.utils.data import Dataset, DataLoader, Subset
from sklearn.model_selection import train_test_split
from scipy.stats import spearmanr
import optuna

import copy

# PyG
from torch_geometric.data import Data, Batch
from torch_geometric.nn import GCNConv, global_mean_pool

# NUPACK
from nupack import Model, pairs

np.set_printoptions(threshold=np.inf)
print("Using PyTorch and PyG.")

########################################
# 1. Data Loading
########################################

def load_branch1_data_AsCas12aRVR():
    """
    Loads (60,4) data from "Feature_CNN1_reduced.txt".
    """
    data_branch1 = []
    current_array = []
    with open("Feature_CNN1_reduced.txt", 'r') as f:
        for line in f:
            line = line.strip()
            if line == "":
                if current_array:
                    data_branch1.append(current_array)
                    current_array = []
            else:
                line = line.replace('[','').replace(']', '')
                row = [float(x.strip().replace(',', '')) for x in line.split()]
                current_array.append(row)
    if current_array:
        data_branch1.append(current_array)
    X_branch1 = np.array(data_branch1)
    X_branch1 = X_branch1.reshape(len(X_branch1), 60, 4)
    return X_branch1

def load_reaction_rates_AsCas12aRVR():
    with open('AsCas12aRVR_indel_frequency_TTTV_filtered.txt', 'r') as f:
        rates = [float(line.strip()) for line in f.readlines()]
    return np.array(rates)


# NUPACK models
from nupack import Model, pairs
my_model_RNA = Model(material='rna', ensemble='nostacking', celsius=37, sodium=0.1, magnesium=0.01)
my_model_DNA = Model(material='dna', ensemble='nostacking', celsius=37, sodium=0.1, magnesium=0.01)

def DNA_reverse_complement(DNA):
    complement = {'A': 'T', 'C': 'G', 'G': 'C', 'T': 'A'}
    return ''.join(complement.get(base, base) for base in reversed(DNA))


########################################
# 3. CNN1 Branches (One hidden MLP)
########################################

class CNNBranch1(nn.Module):
    """
    CNN branch for (60,4).
    """
    def __init__(self, filters, kernel_size, dense_units):
        super().__init__()
        self.conv = nn.Conv1d(4, filters, kernel_size, padding=(kernel_size-1)//2)
        self.pool = nn.MaxPool1d(2)
        self.fc = nn.Linear(filters*30, dense_units)
    def forward(self, x):
        x = x.permute(0,2,1)  # (batch,4,30)
        x = F.relu(self.conv(x))
        x = self.pool(x)      # (batch,filters,30)
        x = x.flatten(start_dim=1)
        x = F.relu(self.fc(x))
        return x


########################################
# 4. Final Fusion Model
########################################

class CNN1OnlyModel(nn.Module):
    def __init__(self, filters1, kernel_size1, dense_units1, final_fc_dim, dropout_rate=0.0):
        super().__init__()
        self.cnn_branch1 = CNNBranch1(filters1, kernel_size1, dense_units1)
        self.dropout_rate = dropout_rate
        self.hidden = nn.Linear(dense_units1, final_fc_dim)
        self.out = nn.Linear(final_fc_dim, 1)

    def forward(self, x1):
        feat1 = self.cnn_branch1(x1)
        merged = F.dropout(feat1, p=self.dropout_rate, training=self.training)
        x = F.relu(self.hidden(merged))
        return self.out(x).view(-1)
        
########################################
# 5. Hybrid Dataset
########################################

class HybridDataset(Dataset):
    def __init__(self, X1, reaction_rates):
        super().__init__()
        self.X1 = X1
        self.reaction_rates = reaction_rates
        self.num_samples = len(reaction_rates)

    def __len__(self):
        return self.num_samples

    def __getitem__(self, idx):
        x1 = torch.tensor(self.X1[idx], dtype=torch.float)
        y_val = torch.tensor([self.reaction_rates[idx]], dtype=torch.float)
        return x1, y_val


def hybrid_collate(batch):
    x1_list, y_list = zip(*batch)
    x1 = torch.stack(x1_list, dim=0)
    y = torch.tensor(y_list, dtype=torch.float).view(-1)
    return x1, y


########################################
# 7. Main Pipeline
########################################

def main_pipeline():
    
    # List all model files you want to evaluate
    model_paths = [
        "../../../CNN1_only_models/trial_0_best_4_dataset_trained_CNN1.pt",
        "../../../CNN1_only_models/trial_1_best_4_dataset_trained_CNN1.pt",
        "../../../CNN1_only_models/trial_2_best_4_dataset_trained_CNN1.pt",
        "../../../CNN1_only_models/trial_3_best_4_dataset_trained_CNN1.pt",
        "../../../CNN1_only_models/trial_4_best_4_dataset_trained_CNN1.pt",
        "../../../CNN1_only_models/trial_5_best_4_dataset_trained_CNN1.pt",
        "../../../CNN1_only_models/trial_6_best_4_dataset_trained_CNN1.pt",
        "../../../CNN1_only_models/trial_7_best_4_dataset_trained_CNN1.pt",
        "../../../CNN1_only_models/trial_8_best_4_dataset_trained_CNN1.pt",
        "../../../CNN1_only_models/trial_9_best_4_dataset_trained_CNN1.pt"
    ]
    
    # final hold-out evaluation
    # load data

    X1_AsCas12aRVR = load_branch1_data_AsCas12aRVR()
    X1_AsCas12aRVR   = np.asarray(X1_AsCas12aRVR)
    X1 = np.concatenate([X1_AsCas12aRVR], axis=0) 

    rates_AsCas12aRVR = load_reaction_rates_AsCas12aRVR()
    rates_AsCas12aRVR   = np.asarray(rates_AsCas12aRVR)
    rates = np.concatenate([rates_AsCas12aRVR], axis=0) 

    hybrid_dataset = HybridDataset(X1, rates)

    # Get unseen AsCas12aRVR dataset
    np.random.seed(42)
    full_indices_AsCas12aRVR = np.arange(len(rates_AsCas12aRVR))
    selected_indices_AsCas12aRVR = np.random.choice(len(full_indices_AsCas12aRVR), size=len(full_indices_AsCas12aRVR), replace=False)
    unseen_indices_AsCas12aRVR = np.setdiff1d(full_indices_AsCas12aRVR, selected_indices_AsCas12aRVR)

    from torch.utils.data import ConcatDataset, DataLoader

    # Evaluate on 10 subsamples from unseen
    unseen_set_AsCas12aRVR = Subset(hybrid_dataset, unseen_indices_AsCas12aRVR)
    
    from torch.utils.data import ConcatDataset, DataLoader

    selected_set_AsCas12aRVR = Subset(hybrid_dataset, selected_indices_AsCas12aRVR)
    trial_loader = DataLoader(selected_set_AsCas12aRVR, batch_size=1, shuffle=False, collate_fn=hybrid_collate)
    
    spearman_results = []

    for model_path in model_paths:
        model = torch.load(model_path, weights_only=False)
        model.eval()

        preds_trial = []
        labels_trial = []

        with torch.no_grad():
            for xx1_b, yy_b in trial_loader:
                p = model(xx1_b)
                preds_trial.append(p.item())
                labels_trial.append(yy_b.item())

        sp_corr, _ = spearmanr(labels_trial, preds_trial)

        print(f"{model_path}: Spearman = {sp_corr}")
        spearman_results.append((model_path, sp_corr))

    with open("Spearman_all_models_4_dataset_CNN1.txt", "w") as f:
        for _, sp_corr in spearman_results:
            f.write(f"{sp_corr}\n")
    

if __name__ == "__main__":
    main_pipeline() 


Using PyTorch and PyG.
../../../CNN1_only_models/trial_0_best_4_dataset_trained_CNN1.pt: Spearman = 0.47735973253402736
../../../CNN1_only_models/trial_1_best_4_dataset_trained_CNN1.pt: Spearman = 0.4772707770487473
../../../CNN1_only_models/trial_2_best_4_dataset_trained_CNN1.pt: Spearman = 0.49376579409164756
../../../CNN1_only_models/trial_3_best_4_dataset_trained_CNN1.pt: Spearman = 0.46674536225278956
../../../CNN1_only_models/trial_4_best_4_dataset_trained_CNN1.pt: Spearman = 0.5188428042193203
../../../CNN1_only_models/trial_5_best_4_dataset_trained_CNN1.pt: Spearman = 0.4908088448348454
../../../CNN1_only_models/trial_6_best_4_dataset_trained_CNN1.pt: Spearman = 0.4752157363959458
../../../CNN1_only_models/trial_7_best_4_dataset_trained_CNN1.pt: Spearman = 0.48689033635209017
../../../CNN1_only_models/trial_8_best_4_dataset_trained_CNN1.pt: Spearman = 0.47787323737121684
../../../CNN1_only_models/trial_9_best_4_dataset_trained_CNN1.pt: Spearman = 0.486959499994779


In [9]:
# 5-dataset model (HTCas9+HT11+RfxCas13d(mmc4+mmc5)+CHANGE_seq)

In [10]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import numpy as np

from torch.utils.data import Dataset, DataLoader, Subset
from sklearn.model_selection import train_test_split
from scipy.stats import spearmanr
import optuna

import copy

# PyG
from torch_geometric.data import Data, Batch
from torch_geometric.nn import GCNConv, global_mean_pool

# NUPACK
from nupack import Model, pairs

np.set_printoptions(threshold=np.inf)
print("Using PyTorch and PyG.")

########################################
# 1. Data Loading
########################################

def load_branch1_data_AsCas12aRVR():
    """
    Loads (60,4) data from "Feature_CNN1_reduced.txt".
    """
    data_branch1 = []
    current_array = []
    with open("Feature_CNN1_reduced.txt", 'r') as f:
        for line in f:
            line = line.strip()
            if line == "":
                if current_array:
                    data_branch1.append(current_array)
                    current_array = []
            else:
                line = line.replace('[','').replace(']', '')
                row = [float(x.strip().replace(',', '')) for x in line.split()]
                current_array.append(row)
    if current_array:
        data_branch1.append(current_array)
    X_branch1 = np.array(data_branch1)
    X_branch1 = X_branch1.reshape(len(X_branch1), 60, 4)
    return X_branch1

def load_reaction_rates_AsCas12aRVR():
    with open('AsCas12aRVR_indel_frequency_TTTV_filtered.txt', 'r') as f:
        rates = [float(line.strip()) for line in f.readlines()]
    return np.array(rates)


# NUPACK models
from nupack import Model, pairs
my_model_RNA = Model(material='rna', ensemble='nostacking', celsius=37, sodium=0.1, magnesium=0.01)
my_model_DNA = Model(material='dna', ensemble='nostacking', celsius=37, sodium=0.1, magnesium=0.01)

def DNA_reverse_complement(DNA):
    complement = {'A': 'T', 'C': 'G', 'G': 'C', 'T': 'A'}
    return ''.join(complement.get(base, base) for base in reversed(DNA))


########################################
# 3. CNN1 Branches (One hidden MLP)
########################################

class CNNBranch1(nn.Module):
    """
    CNN branch for (60,4).
    """
    def __init__(self, filters, kernel_size, dense_units):
        super().__init__()
        self.conv = nn.Conv1d(4, filters, kernel_size, padding=(kernel_size-1)//2)
        self.pool = nn.MaxPool1d(2)
        self.fc = nn.Linear(filters*30, dense_units)
    def forward(self, x):
        x = x.permute(0,2,1)  # (batch,4,30)
        x = F.relu(self.conv(x))
        x = self.pool(x)      # (batch,filters,30)
        x = x.flatten(start_dim=1)
        x = F.relu(self.fc(x))
        return x


########################################
# 4. Final Fusion Model
########################################

class CNN1OnlyModel(nn.Module):
    def __init__(self, filters1, kernel_size1, dense_units1, final_fc_dim, dropout_rate=0.0):
        super().__init__()
        self.cnn_branch1 = CNNBranch1(filters1, kernel_size1, dense_units1)
        self.dropout_rate = dropout_rate
        self.hidden = nn.Linear(dense_units1, final_fc_dim)
        self.out = nn.Linear(final_fc_dim, 1)

    def forward(self, x1):
        feat1 = self.cnn_branch1(x1)
        merged = F.dropout(feat1, p=self.dropout_rate, training=self.training)
        x = F.relu(self.hidden(merged))
        return self.out(x).view(-1)
        
########################################
# 5. Hybrid Dataset
########################################

class HybridDataset(Dataset):
    def __init__(self, X1, reaction_rates):
        super().__init__()
        self.X1 = X1
        self.reaction_rates = reaction_rates
        self.num_samples = len(reaction_rates)

    def __len__(self):
        return self.num_samples

    def __getitem__(self, idx):
        x1 = torch.tensor(self.X1[idx], dtype=torch.float)
        y_val = torch.tensor([self.reaction_rates[idx]], dtype=torch.float)
        return x1, y_val


def hybrid_collate(batch):
    x1_list, y_list = zip(*batch)
    x1 = torch.stack(x1_list, dim=0)
    y = torch.tensor(y_list, dtype=torch.float).view(-1)
    return x1, y


########################################
# 7. Main Pipeline
########################################

def main_pipeline():
    
    # List all model files you want to evaluate
    model_paths = [
        "../../../CNN1_only_models/trial_0_best_5_dataset_trained_CNN1.pt",
        "../../../CNN1_only_models/trial_1_best_5_dataset_trained_CNN1.pt",
        "../../../CNN1_only_models/trial_2_best_5_dataset_trained_CNN1.pt",
        "../../../CNN1_only_models/trial_3_best_5_dataset_trained_CNN1.pt",
        "../../../CNN1_only_models/trial_4_best_5_dataset_trained_CNN1.pt",
        "../../../CNN1_only_models/trial_5_best_5_dataset_trained_CNN1.pt",
        "../../../CNN1_only_models/trial_6_best_5_dataset_trained_CNN1.pt",
        "../../../CNN1_only_models/trial_7_best_5_dataset_trained_CNN1.pt",
        "../../../CNN1_only_models/trial_8_best_5_dataset_trained_CNN1.pt",
        "../../../CNN1_only_models/trial_9_best_5_dataset_trained_CNN1.pt"
    ]
    
    # final hold-out evaluation
    # load data

    X1_AsCas12aRVR = load_branch1_data_AsCas12aRVR()
    X1_AsCas12aRVR   = np.asarray(X1_AsCas12aRVR)
    X1 = np.concatenate([X1_AsCas12aRVR], axis=0) 

    rates_AsCas12aRVR = load_reaction_rates_AsCas12aRVR()
    rates_AsCas12aRVR   = np.asarray(rates_AsCas12aRVR)
    rates = np.concatenate([rates_AsCas12aRVR], axis=0) 

    hybrid_dataset = HybridDataset(X1, rates)

    # Get unseen AsCas12aRVR dataset
    np.random.seed(42)
    full_indices_AsCas12aRVR = np.arange(len(rates_AsCas12aRVR))
    selected_indices_AsCas12aRVR = np.random.choice(len(full_indices_AsCas12aRVR), size=len(full_indices_AsCas12aRVR), replace=False)
    unseen_indices_AsCas12aRVR = np.setdiff1d(full_indices_AsCas12aRVR, selected_indices_AsCas12aRVR)

    from torch.utils.data import ConcatDataset, DataLoader

    # Evaluate on 10 subsamples from unseen
    unseen_set_AsCas12aRVR = Subset(hybrid_dataset, unseen_indices_AsCas12aRVR)
    
    from torch.utils.data import ConcatDataset, DataLoader

    selected_set_AsCas12aRVR = Subset(hybrid_dataset, selected_indices_AsCas12aRVR)
    trial_loader = DataLoader(selected_set_AsCas12aRVR, batch_size=1, shuffle=False, collate_fn=hybrid_collate)
    
    spearman_results = []

    for model_path in model_paths:
        model = torch.load(model_path, weights_only=False)
        model.eval()

        preds_trial = []
        labels_trial = []

        with torch.no_grad():
            for xx1_b, yy_b in trial_loader:
                p = model(xx1_b)
                preds_trial.append(p.item())
                labels_trial.append(yy_b.item())

        sp_corr, _ = spearmanr(labels_trial, preds_trial)

        print(f"{model_path}: Spearman = {sp_corr}")
        spearman_results.append((model_path, sp_corr))

    with open("Spearman_all_models_5_dataset_CNN1.txt", "w") as f:
        for _, sp_corr in spearman_results:
            f.write(f"{sp_corr}\n")
    

if __name__ == "__main__":
    main_pipeline() 


Using PyTorch and PyG.
../../../CNN1_only_models/trial_0_best_5_dataset_trained_CNN1.pt: Spearman = 0.49319151101181213
../../../CNN1_only_models/trial_1_best_5_dataset_trained_CNN1.pt: Spearman = 0.4813651523128229
../../../CNN1_only_models/trial_2_best_5_dataset_trained_CNN1.pt: Spearman = 0.47071365485960553
../../../CNN1_only_models/trial_3_best_5_dataset_trained_CNN1.pt: Spearman = 0.4947756150205343
../../../CNN1_only_models/trial_4_best_5_dataset_trained_CNN1.pt: Spearman = 0.4965732758766447
../../../CNN1_only_models/trial_5_best_5_dataset_trained_CNN1.pt: Spearman = 0.49471620552437584
../../../CNN1_only_models/trial_6_best_5_dataset_trained_CNN1.pt: Spearman = 0.5024311577362645
../../../CNN1_only_models/trial_7_best_5_dataset_trained_CNN1.pt: Spearman = 0.4731371474111145
../../../CNN1_only_models/trial_8_best_5_dataset_trained_CNN1.pt: Spearman = 0.4357549635169017
../../../CNN1_only_models/trial_9_best_5_dataset_trained_CNN1.pt: Spearman = 0.47632135036464957


In [11]:
# 6-dataset model (HTCas9+HT11+RfxCas13d(mmc4+mmc5)+CHANGE_seq+TIGER)

In [12]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import numpy as np

from torch.utils.data import Dataset, DataLoader, Subset
from sklearn.model_selection import train_test_split
from scipy.stats import spearmanr
import optuna

import copy

# PyG
from torch_geometric.data import Data, Batch
from torch_geometric.nn import GCNConv, global_mean_pool

# NUPACK
from nupack import Model, pairs

np.set_printoptions(threshold=np.inf)
print("Using PyTorch and PyG.")

########################################
# 1. Data Loading
########################################

def load_branch1_data_AsCas12aRVR():
    """
    Loads (60,4) data from "Feature_CNN1_reduced.txt".
    """
    data_branch1 = []
    current_array = []
    with open("Feature_CNN1_reduced.txt", 'r') as f:
        for line in f:
            line = line.strip()
            if line == "":
                if current_array:
                    data_branch1.append(current_array)
                    current_array = []
            else:
                line = line.replace('[','').replace(']', '')
                row = [float(x.strip().replace(',', '')) for x in line.split()]
                current_array.append(row)
    if current_array:
        data_branch1.append(current_array)
    X_branch1 = np.array(data_branch1)
    X_branch1 = X_branch1.reshape(len(X_branch1), 60, 4)
    return X_branch1

def load_reaction_rates_AsCas12aRVR():
    with open('AsCas12aRVR_indel_frequency_TTTV_filtered.txt', 'r') as f:
        rates = [float(line.strip()) for line in f.readlines()]
    return np.array(rates)


# NUPACK models
from nupack import Model, pairs
my_model_RNA = Model(material='rna', ensemble='nostacking', celsius=37, sodium=0.1, magnesium=0.01)
my_model_DNA = Model(material='dna', ensemble='nostacking', celsius=37, sodium=0.1, magnesium=0.01)

def DNA_reverse_complement(DNA):
    complement = {'A': 'T', 'C': 'G', 'G': 'C', 'T': 'A'}
    return ''.join(complement.get(base, base) for base in reversed(DNA))


########################################
# 3. CNN1 Branches (One hidden MLP)
########################################

class CNNBranch1(nn.Module):
    """
    CNN branch for (60,4).
    """
    def __init__(self, filters, kernel_size, dense_units):
        super().__init__()
        self.conv = nn.Conv1d(4, filters, kernel_size, padding=(kernel_size-1)//2)
        self.pool = nn.MaxPool1d(2)
        self.fc = nn.Linear(filters*30, dense_units)
    def forward(self, x):
        x = x.permute(0,2,1)  # (batch,4,30)
        x = F.relu(self.conv(x))
        x = self.pool(x)      # (batch,filters,30)
        x = x.flatten(start_dim=1)
        x = F.relu(self.fc(x))
        return x


########################################
# 4. Final Fusion Model
########################################

class CNN1OnlyModel(nn.Module):
    def __init__(self, filters1, kernel_size1, dense_units1, final_fc_dim, dropout_rate=0.0):
        super().__init__()
        self.cnn_branch1 = CNNBranch1(filters1, kernel_size1, dense_units1)
        self.dropout_rate = dropout_rate
        self.hidden = nn.Linear(dense_units1, final_fc_dim)
        self.out = nn.Linear(final_fc_dim, 1)

    def forward(self, x1):
        feat1 = self.cnn_branch1(x1)
        merged = F.dropout(feat1, p=self.dropout_rate, training=self.training)
        x = F.relu(self.hidden(merged))
        return self.out(x).view(-1)
        
########################################
# 5. Hybrid Dataset
########################################

class HybridDataset(Dataset):
    def __init__(self, X1, reaction_rates):
        super().__init__()
        self.X1 = X1
        self.reaction_rates = reaction_rates
        self.num_samples = len(reaction_rates)

    def __len__(self):
        return self.num_samples

    def __getitem__(self, idx):
        x1 = torch.tensor(self.X1[idx], dtype=torch.float)
        y_val = torch.tensor([self.reaction_rates[idx]], dtype=torch.float)
        return x1, y_val


def hybrid_collate(batch):
    x1_list, y_list = zip(*batch)
    x1 = torch.stack(x1_list, dim=0)
    y = torch.tensor(y_list, dtype=torch.float).view(-1)
    return x1, y


########################################
# 7. Main Pipeline
########################################

def main_pipeline():
    
    # List all model files you want to evaluate
    model_paths = [
        "../../../CNN1_only_models/trial_0_best_6_dataset_trained_CNN1.pt",
        "../../../CNN1_only_models/trial_1_best_6_dataset_trained_CNN1.pt",
        "../../../CNN1_only_models/trial_2_best_6_dataset_trained_CNN1.pt",
        "../../../CNN1_only_models/trial_3_best_6_dataset_trained_CNN1.pt",
        "../../../CNN1_only_models/trial_4_best_6_dataset_trained_CNN1.pt",
        "../../../CNN1_only_models/trial_5_best_6_dataset_trained_CNN1.pt",
        "../../../CNN1_only_models/trial_6_best_6_dataset_trained_CNN1.pt",
        "../../../CNN1_only_models/trial_7_best_6_dataset_trained_CNN1.pt",
        "../../../CNN1_only_models/trial_8_best_6_dataset_trained_CNN1.pt",
        "../../../CNN1_only_models/trial_9_best_6_dataset_trained_CNN1.pt"
    ]
    
    # final hold-out evaluation
    # load data

    X1_AsCas12aRVR = load_branch1_data_AsCas12aRVR()
    X1_AsCas12aRVR   = np.asarray(X1_AsCas12aRVR)
    X1 = np.concatenate([X1_AsCas12aRVR], axis=0) 

    rates_AsCas12aRVR = load_reaction_rates_AsCas12aRVR()
    rates_AsCas12aRVR   = np.asarray(rates_AsCas12aRVR)
    rates = np.concatenate([rates_AsCas12aRVR], axis=0) 

    hybrid_dataset = HybridDataset(X1, rates)

    # Get unseen AsCas12aRVR dataset
    np.random.seed(42)
    full_indices_AsCas12aRVR = np.arange(len(rates_AsCas12aRVR))
    selected_indices_AsCas12aRVR = np.random.choice(len(full_indices_AsCas12aRVR), size=len(full_indices_AsCas12aRVR), replace=False)
    unseen_indices_AsCas12aRVR = np.setdiff1d(full_indices_AsCas12aRVR, selected_indices_AsCas12aRVR)

    from torch.utils.data import ConcatDataset, DataLoader

    # Evaluate on 10 subsamples from unseen
    unseen_set_AsCas12aRVR = Subset(hybrid_dataset, unseen_indices_AsCas12aRVR)
    
    from torch.utils.data import ConcatDataset, DataLoader

    selected_set_AsCas12aRVR = Subset(hybrid_dataset, selected_indices_AsCas12aRVR)
    trial_loader = DataLoader(selected_set_AsCas12aRVR, batch_size=1, shuffle=False, collate_fn=hybrid_collate)
    
    spearman_results = []

    for model_path in model_paths:
        model = torch.load(model_path, weights_only=False)
        model.eval()

        preds_trial = []
        labels_trial = []

        with torch.no_grad():
            for xx1_b, yy_b in trial_loader:
                p = model(xx1_b)
                preds_trial.append(p.item())
                labels_trial.append(yy_b.item())

        sp_corr, _ = spearmanr(labels_trial, preds_trial)

        print(f"{model_path}: Spearman = {sp_corr}")
        spearman_results.append((model_path, sp_corr))

    with open("Spearman_all_models_6_dataset_CNN1.txt", "w") as f:
        for _, sp_corr in spearman_results:
            f.write(f"{sp_corr}\n")
    

if __name__ == "__main__":
    main_pipeline() 


Using PyTorch and PyG.
../../../CNN1_only_models/trial_0_best_6_dataset_trained_CNN1.pt: Spearman = 0.4854919498961969
../../../CNN1_only_models/trial_1_best_6_dataset_trained_CNN1.pt: Spearman = 0.47660104520682045
../../../CNN1_only_models/trial_2_best_6_dataset_trained_CNN1.pt: Spearman = 0.48529209831441716
../../../CNN1_only_models/trial_3_best_6_dataset_trained_CNN1.pt: Spearman = 0.47028087827542686
../../../CNN1_only_models/trial_4_best_6_dataset_trained_CNN1.pt: Spearman = 0.45917985386163584
../../../CNN1_only_models/trial_5_best_6_dataset_trained_CNN1.pt: Spearman = 0.48401925684618874
../../../CNN1_only_models/trial_6_best_6_dataset_trained_CNN1.pt: Spearman = 0.4824036316183095
../../../CNN1_only_models/trial_7_best_6_dataset_trained_CNN1.pt: Spearman = 0.4667624617446714
../../../CNN1_only_models/trial_8_best_6_dataset_trained_CNN1.pt: Spearman = 0.47471937372458844
../../../CNN1_only_models/trial_9_best_6_dataset_trained_CNN1.pt: Spearman = 0.47924807118030194


In [13]:
# 7-dataset model (HTCas9+HT11+RfxCas13d(mmc4+mmc5)+CHANGE_seq+TIGER+iMeta)

In [14]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import numpy as np

from torch.utils.data import Dataset, DataLoader, Subset
from sklearn.model_selection import train_test_split
from scipy.stats import spearmanr
import optuna

import copy

# PyG
from torch_geometric.data import Data, Batch
from torch_geometric.nn import GCNConv, global_mean_pool

# NUPACK
from nupack import Model, pairs

np.set_printoptions(threshold=np.inf)
print("Using PyTorch and PyG.")

########################################
# 1. Data Loading
########################################

def load_branch1_data_AsCas12aRVR():
    """
    Loads (60,4) data from "Feature_CNN1_reduced.txt".
    """
    data_branch1 = []
    current_array = []
    with open("Feature_CNN1_reduced.txt", 'r') as f:
        for line in f:
            line = line.strip()
            if line == "":
                if current_array:
                    data_branch1.append(current_array)
                    current_array = []
            else:
                line = line.replace('[','').replace(']', '')
                row = [float(x.strip().replace(',', '')) for x in line.split()]
                current_array.append(row)
    if current_array:
        data_branch1.append(current_array)
    X_branch1 = np.array(data_branch1)
    X_branch1 = X_branch1.reshape(len(X_branch1), 60, 4)
    return X_branch1

def load_reaction_rates_AsCas12aRVR():
    with open('AsCas12aRVR_indel_frequency_TTTV_filtered.txt', 'r') as f:
        rates = [float(line.strip()) for line in f.readlines()]
    return np.array(rates)


# NUPACK models
from nupack import Model, pairs
my_model_RNA = Model(material='rna', ensemble='nostacking', celsius=37, sodium=0.1, magnesium=0.01)
my_model_DNA = Model(material='dna', ensemble='nostacking', celsius=37, sodium=0.1, magnesium=0.01)

def DNA_reverse_complement(DNA):
    complement = {'A': 'T', 'C': 'G', 'G': 'C', 'T': 'A'}
    return ''.join(complement.get(base, base) for base in reversed(DNA))


########################################
# 3. CNN1 Branches (One hidden MLP)
########################################

class CNNBranch1(nn.Module):
    """
    CNN branch for (60,4).
    """
    def __init__(self, filters, kernel_size, dense_units):
        super().__init__()
        self.conv = nn.Conv1d(4, filters, kernel_size, padding=(kernel_size-1)//2)
        self.pool = nn.MaxPool1d(2)
        self.fc = nn.Linear(filters*30, dense_units)
    def forward(self, x):
        x = x.permute(0,2,1)  # (batch,4,30)
        x = F.relu(self.conv(x))
        x = self.pool(x)      # (batch,filters,30)
        x = x.flatten(start_dim=1)
        x = F.relu(self.fc(x))
        return x


########################################
# 4. Final Fusion Model
########################################

class CNN1OnlyModel(nn.Module):
    def __init__(self, filters1, kernel_size1, dense_units1, final_fc_dim, dropout_rate=0.0):
        super().__init__()
        self.cnn_branch1 = CNNBranch1(filters1, kernel_size1, dense_units1)
        self.dropout_rate = dropout_rate
        self.hidden = nn.Linear(dense_units1, final_fc_dim)
        self.out = nn.Linear(final_fc_dim, 1)

    def forward(self, x1):
        feat1 = self.cnn_branch1(x1)
        merged = F.dropout(feat1, p=self.dropout_rate, training=self.training)
        x = F.relu(self.hidden(merged))
        return self.out(x).view(-1)
        
########################################
# 5. Hybrid Dataset
########################################

class HybridDataset(Dataset):
    def __init__(self, X1, reaction_rates):
        super().__init__()
        self.X1 = X1
        self.reaction_rates = reaction_rates
        self.num_samples = len(reaction_rates)

    def __len__(self):
        return self.num_samples

    def __getitem__(self, idx):
        x1 = torch.tensor(self.X1[idx], dtype=torch.float)
        y_val = torch.tensor([self.reaction_rates[idx]], dtype=torch.float)
        return x1, y_val


def hybrid_collate(batch):
    x1_list, y_list = zip(*batch)
    x1 = torch.stack(x1_list, dim=0)
    y = torch.tensor(y_list, dtype=torch.float).view(-1)
    return x1, y


########################################
# 7. Main Pipeline
########################################

def main_pipeline():
    
    # List all model files you want to evaluate
    model_paths = [
        "../../../CNN1_only_models/trial_0_best_7_dataset_trained_CNN1.pt",
        "../../../CNN1_only_models/trial_1_best_7_dataset_trained_CNN1.pt",
        "../../../CNN1_only_models/trial_2_best_7_dataset_trained_CNN1.pt",
        "../../../CNN1_only_models/trial_3_best_7_dataset_trained_CNN1.pt",
        "../../../CNN1_only_models/trial_4_best_7_dataset_trained_CNN1.pt",
        "../../../CNN1_only_models/trial_5_best_7_dataset_trained_CNN1.pt",
        "../../../CNN1_only_models/trial_6_best_7_dataset_trained_CNN1.pt",
        "../../../CNN1_only_models/trial_7_best_7_dataset_trained_CNN1.pt",
        "../../../CNN1_only_models/trial_8_best_7_dataset_trained_CNN1.pt",
        "../../../CNN1_only_models/trial_9_best_7_dataset_trained_CNN1.pt"
    ]
    
    # final hold-out evaluation
    # load data

    X1_AsCas12aRVR = load_branch1_data_AsCas12aRVR()
    X1_AsCas12aRVR   = np.asarray(X1_AsCas12aRVR)
    X1 = np.concatenate([X1_AsCas12aRVR], axis=0) 

    rates_AsCas12aRVR = load_reaction_rates_AsCas12aRVR()
    rates_AsCas12aRVR   = np.asarray(rates_AsCas12aRVR)
    rates = np.concatenate([rates_AsCas12aRVR], axis=0) 

    hybrid_dataset = HybridDataset(X1, rates)

    # Get unseen AsCas12aRVR dataset
    np.random.seed(42)
    full_indices_AsCas12aRVR = np.arange(len(rates_AsCas12aRVR))
    selected_indices_AsCas12aRVR = np.random.choice(len(full_indices_AsCas12aRVR), size=len(full_indices_AsCas12aRVR), replace=False)
    unseen_indices_AsCas12aRVR = np.setdiff1d(full_indices_AsCas12aRVR, selected_indices_AsCas12aRVR)

    from torch.utils.data import ConcatDataset, DataLoader

    # Evaluate on 10 subsamples from unseen
    unseen_set_AsCas12aRVR = Subset(hybrid_dataset, unseen_indices_AsCas12aRVR)
    
    from torch.utils.data import ConcatDataset, DataLoader

    selected_set_AsCas12aRVR = Subset(hybrid_dataset, selected_indices_AsCas12aRVR)
    trial_loader = DataLoader(selected_set_AsCas12aRVR, batch_size=1, shuffle=False, collate_fn=hybrid_collate)
    
    spearman_results = []

    for model_path in model_paths:
        model = torch.load(model_path, weights_only=False)
        model.eval()

        preds_trial = []
        labels_trial = []

        with torch.no_grad():
            for xx1_b, yy_b in trial_loader:
                p = model(xx1_b)
                preds_trial.append(p.item())
                labels_trial.append(yy_b.item())

        sp_corr, _ = spearmanr(labels_trial, preds_trial)

        print(f"{model_path}: Spearman = {sp_corr}")
        spearman_results.append((model_path, sp_corr))

    with open("Spearman_all_models_7_dataset_CNN1.txt", "w") as f:
        for _, sp_corr in spearman_results:
            f.write(f"{sp_corr}\n")
    

if __name__ == "__main__":
    main_pipeline() 


Using PyTorch and PyG.
../../../CNN1_only_models/trial_0_best_7_dataset_trained_CNN1.pt: Spearman = 0.5129420286583429
../../../CNN1_only_models/trial_1_best_7_dataset_trained_CNN1.pt: Spearman = 0.48378936159117164
../../../CNN1_only_models/trial_2_best_7_dataset_trained_CNN1.pt: Spearman = 0.4883528447677835
../../../CNN1_only_models/trial_3_best_7_dataset_trained_CNN1.pt: Spearman = 0.48920798508074703
../../../CNN1_only_models/trial_4_best_7_dataset_trained_CNN1.pt: Spearman = 0.46628708880600334
../../../CNN1_only_models/trial_5_best_7_dataset_trained_CNN1.pt: Spearman = 0.4747471563195762
../../../CNN1_only_models/trial_6_best_7_dataset_trained_CNN1.pt: Spearman = 0.46307487908082484
../../../CNN1_only_models/trial_7_best_7_dataset_trained_CNN1.pt: Spearman = 0.4978186833751269
../../../CNN1_only_models/trial_8_best_7_dataset_trained_CNN1.pt: Spearman = 0.488887249916588
../../../CNN1_only_models/trial_9_best_7_dataset_trained_CNN1.pt: Spearman = 0.4881491637748532
